# From AI Transcription to Structured Historical Data: A Gemini API Workflow

This notebook accompanies the **Digital History Pedagogy Toolkit** tutorial.

It begins with a **reviewed historical transcription** and uses the Gemini API to extract:

1. People
2. Places
3. Dates and times
4. Organizations / offices
5. Actions / events explicitly stated in the text

> **Why Gemini?** This notebook uses the Gemini API Free Tier because it provides an accessible way to experiment with AI-assisted structured extraction. The workflow itself can be adapted to other AI models and platforms.

**Last reviewed:** October 2026


## 1. Input Options

You can use either:

- the Toolkit's `sample_transcriptions.csv`, in which each row represents one reviewed archival page and `record_id` identifies the record; or
- `archival_transcriptions.csv` from the previous Gemini transcription tutorial, in which `image_file` identifies the source image/page.

Both datasets must contain a `transcription` column.

The sample CSV combines page-level transcriptions that may originally have existed as separate `.txt` files into one table with one row per page.


## 2. Install and Import Packages

In [ ]:
%pip install -U google-genai pydantic pandas

In [ ]:
from google import genai
from getpass import getpass
from pydantic import BaseModel, Field
from datetime import datetime, timezone
from pathlib import Path
import json
import pandas as pd

## 3. Connect to Gemini

Never place an API key directly in a notebook you plan to share.


In [ ]:
GEMINI_API_KEY = getpass("Enter your Gemini API key: ")

In [ ]:
client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
MODEL = "gemini-3.8-flash"

## 4. Define the Extraction Schema

In [ ]:
class HistoricalExtraction(BaseModel):
    people: list[str] = Field(
        description="People explicitly named in the transcription."
    )
    places: list[str] = Field(
        description="Places explicitly named in the transcription."
    )
    dates_times: list[str] = Field(
        description="Dates and times explicitly stated in the transcription."
    )
    organizations_offices: list[str] = Field(
        description="Organizations or offices explicitly named in the transcription."
    )
    actions_events: list[str] = Field(
        description="Actions or events explicitly stated in the transcription."
    )

## 5. Define the Extraction Instructions

In [ ]:
EXTRACTION_PROMPT = """
Extract structured historical information from the transcription below.

Return only information explicitly present in the transcription.

Use these categories:
- people
- places
- dates and times
- organizations or offices
- actions or events explicitly stated in the text

Rules:
1. Do not add information from outside the transcription.
2. Do not expand names using outside knowledge.
3. Do not modernize historical place names.
4. Do not infer missing dates, identities, motives, causes, or relationships.
5. Preserve the wording of names and places as it appears in the transcription.
6. If a category contains no information, return an empty list.
"""

## 6. Load Your Dataset

Choose **one** file.

For local Jupyter:
```python
DATA_PATH = Path("sample_transcriptions.csv")
```
or:
```python
DATA_PATH = Path("archival_transcriptions.csv")
```

For Colab, upload the file first and use `/content/...`.


In [ ]:
# Choose ONE path before continuing.

# Toolkit sample dataset:
# DATA_PATH = Path("sample_transcriptions.csv")

# Previous Gemini tutorial dataset:
# DATA_PATH = Path("archival_transcriptions.csv")

# Colab examples:
# DATA_PATH = Path("/content/sample_transcriptions.csv")
# DATA_PATH = Path("/content/archival_transcriptions.csv")

In [ ]:
if "DATA_PATH" not in globals():
    raise NameError(
        "Set DATA_PATH first by uncommenting one of the path options above."
    )

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Upload or place the CSV file first."
    )

transcriptions_df = pd.read_csv(DATA_PATH)
transcriptions_df

The next cell accepts either `record_id` or `image_file` as the record identifier.


In [ ]:
if "record_id" in transcriptions_df.columns:
    ID_COLUMN = "record_id"
elif "image_file" in transcriptions_df.columns:
    ID_COLUMN = "image_file"
else:
    raise ValueError(
        "The dataset must contain either 'record_id' or 'image_file'."
    )

if "transcription" not in transcriptions_df.columns:
    raise ValueError(
        "The dataset must contain a 'transcription' column."
    )

print(f"Using '{ID_COLUMN}' as the record identifier.")

## 7. Start with One Transcription

Before processing the whole dataset, inspect and test one record.


In [ ]:
TRANSCRIPTION = transcriptions_df.loc[0, "transcription"]

print(TRANSCRIPTION)

## 8. Ask Gemini for Structured Output

In [ ]:
interaction = client.interactions.create(
    model=MODEL,
    input=f"""
{EXTRACTION_PROMPT}

TRANSCRIPTION:
{TRANSCRIPTION}
""",
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": HistoricalExtraction.model_json_schema()
    },
)

In [ ]:
extraction = HistoricalExtraction.model_validate_json(
    interaction.output_text
)

print(extraction.model_dump_json(indent=2))

## 9. Your Task: Evaluate the Extraction

Compare the structured output directly with the transcription.

Look for:

- **Correct** items
- **Omitted** items
- **Misclassified** items
- **Added by the model** items

Record observations about what worked well and what required interpretation.


## 10. Create a Reusable Extraction Function

In [ ]:
def extract_historical_data(transcription, model=MODEL):

    interaction = client.interactions.create(
        model=model,
        input=f"""
{EXTRACTION_PROMPT}

TRANSCRIPTION:
{transcription}
""",
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": HistoricalExtraction.model_json_schema()
        },
    )

    return HistoricalExtraction.model_validate_json(
        interaction.output_text
    )

## 11. Process Several Transcriptions

In [ ]:
batch_results = []

In [ ]:
for _, row in transcriptions_df.iterrows():

    print(f"Processing: {row[ID_COLUMN]}")

    try:
        extraction = extract_historical_data(
            row["transcription"]
        )

        result = extraction.model_dump()

        result["record_id"] = row[ID_COLUMN]
        result["transcription"] = row["transcription"]
        result["model"] = MODEL
        result["processed_at_utc"] = datetime.now(
            timezone.utc
        ).isoformat()
        result["review_status"] = ""
        result["review_notes"] = ""

        batch_results.append(result)

    except Exception as e:

        batch_results.append({
            "record_id": row[ID_COLUMN],
            "transcription": row["transcription"],
            "people": [],
            "places": [],
            "dates_times": [],
            "organizations_offices": [],
            "actions_events": [],
            "model": MODEL,
            "processed_at_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "review_status": "API error",
            "review_notes": str(e)
        })

`extraction.model_dump()` converts the validated result into a regular Python dictionary so that we can add metadata such as the record ID, model, and review fields.

For larger datasets, check the provider's current API rate limits before scaling up.


## 12. Save JSON

In [ ]:
with open(
    "structured_historical_data.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        batch_results,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved structured_historical_data.json")

## 13. Save a Spreadsheet-Friendly CSV

In [ ]:
csv_rows = []

for result in batch_results:
    csv_rows.append({
        "record_id": result["record_id"],
        "transcription": result["transcription"],
        "people": "; ".join(result["people"]),
        "places": "; ".join(result["places"]),
        "dates_times": "; ".join(result["dates_times"]),
        "organizations_offices": "; ".join(
            result["organizations_offices"]
        ),
        "actions_events": "; ".join(
            result["actions_events"]
        ),
        "model": result["model"],
        "processed_at_utc": result["processed_at_utc"],
        "review_status": result["review_status"],
        "review_notes": result["review_notes"]
    })

structured_df = pd.DataFrame(csv_rows)
structured_df

In [ ]:
structured_df.to_csv(
    "structured_historical_data.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved structured_historical_data.csv")

## 14. What Changes at Scale?

With five records, it is still practical to inspect every output manually.

With hundreds or thousands of records, exhaustive inspection becomes increasingly costly. Larger projects therefore need a quality-control strategy: representative sampling, recurring error categories, field-specific review, and appropriate error measurement.

> **Scaling does not eliminate the need for human evaluation. It changes how that evaluation has to be designed.**


## 15. Adapting the Workflow

The same general workflow can be adapted to other AI platforms:

`reviewed transcription → define extraction categories → structured output → human evaluation → dataset`

The provider-specific code may change, but the historical-method decisions remain the same.
